<a href="https://colab.research.google.com/github/ashsiendo/DATA-201-Labs/blob/main/Lab02_week2_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro to Python

*Added comments include Python-R comparisons and notes to self.*

### Part 1 - Python Warm-Up


In [ ]:
#variables and their values
age = 20
study_hours = 6
midterm_grade = 82
student_name = "Sara"
passed_midterm = True

print(age)
print(study_hours)
print(midterm_grade)

In [ ]:
#Find the data type- each variable
#type() similar to checking class() in R
print(type(age))
print(type(student_name))
print(type(passed_midterm))

### Part 2 — NumPy Arrays

In [ ]:
#NumPy is a Python library for working efficiently with numerical data.
#NumPy used for efficient numerical arrays and calculations
import numpy as np

ages = np.array([18, 21, 25, 32, 40]) # np.array() creates NumPy array, similar to numeric vector in R
print(ages)
print(type(ages))


age=(1,2,3) # parentheses create tuple here, not NumPy array
print(age)
print(type(age))

**Data Types**

* **Tuple:** Heterogeneous. It can store a mix of different data types (strings, numbers, booleans) together.
* **Array:** Homogeneous. Every element must be of the exact same data type.

**Mutability**

* **Tuple:** Immutable. Elements cannot be altered, added, or removed after creation.
* **Array:** Mutable. Individual elements can be modified, though total size is often fixed.

**Common Analogy**

* **Tuple:** Works like a database record or a lightweight struct with fixed positions.
* **Array:** Works like a mathematical vector or a simple inventory list.

*In other words, tuple ≠ NumPy array*


In [ ]:
# error because tuples don't have .mean() method
age.mean()

In [ ]:
ages.mean()

In [ ]:
ages.min()

In [ ]:
ages.max()

In [ ]:
income = np.array([30000, 45000, 52000, 75000, 120000])
income.mean()

### Part 3 — Selecting Values

In [ ]:
ages

In [ ]:
#selecting the first value - python starts at 0, so [0] selects first value
ages[0]

In [ ]:
#selecting second value to the fourth value
# slice from index 1 up to, but not including, index 4
ages[1:4]

In [ ]:
ages[4]

In [ ]:
# select multiple specific positions using list of indices
ages[[1,3]]

In [ ]:
#select the first and the fourth value from 'income'
income[[0,3]]


***Note to self and a reminder that unlike R, Python starts indexing at 0***

### Part 4 - Working With a Real Dataset

Upload the Auto dataset to the "Datasets" folder inside your Data_201 folder in Google Drive.

In Google Colab, connect your Google Drive by running:

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Define the file path

In [ ]:
import pandas as pd
Auto = pd.read_csv("/content/drive/MyDrive/Data_201/Datasets/Auto.csv")

In [ ]:
Auto.head()

- What does each row represent?
- What do columns represent?
- Which variables look quantitative?
- Which might actually represent categories?

In [ ]:
Auto.shape

In [ ]:
Auto["origin"].mean() #does this make sense?

In [ ]:
Auto.columns

### Part 5 — Data Quality and Missing Values

In [ ]:
Auto.isna().sum()

### Part 6 — Selecting Features and Summarizing Data

In [ ]:
Auto["horsepower"]

In [ ]:
Auto[["horsepower", "mpg"]]

In [ ]:
cars_small = Auto[["mpg", "horsepower", "weight"]]
cars_small.head()

Find the:

- average MPG
- minimum weight

In [ ]:
cars_small["mpg"].mean()

In [ ]:
Auto["weight"].min()

In [ ]:
#Now try to find the max of horsepower. What does the result mean?

Auto["horsepower"].max()

The missing horsepower values are stored as the literal character "?"
Thus, pandas does not automatically recognize them as NaN.

That is why 'Auto.isna().sum()' showed 0!

In [ ]:
cars_small["horsepower"] = cars_small["horsepower"].replace("?", pd.NA)

In [ ]:
#Then convert the column to numeric
cars_small["horsepower"] = pd.to_numeric(cars_small["horsepower"])

In [ ]:
cars_small["horsepower"].max()

### Part 7 — One-Hot Encoding


Machine-learning algorithms generally need numeric input. But some numbers in a dataset may actually represent categories rather than quantities.

**For Example**: The origin variable in Auto uses numbers to represent where the vehicle came from.

1 = USA
2 = Europe
3 = Japan

In [ ]:
Auto["origin"].value_counts()

In [ ]:
#Change the code to meaningful labels
Auto["origin_label"] = Auto["origin"].map({
    1: "USA",
    2: "Europe",
    3: "Japan"
})

In [ ]:
Auto[["origin", "origin_label"]].head()

In [ ]:
# Now one-hot encode
pd.get_dummies(Auto["origin_label"], dtype=int)

Instead of pretending that USA, Europe, and Japan have a numerical order, Python creates a separate 0/1 variable for each category. Because those numbers are just labels. They do not represent magnitude or order.

### Part 8 — Scaling

In [ ]:
# fixing "horsepower" in Auto
# replace "?" with missing values, then convert horsepower from text to numeric
Auto["horsepower"] = Auto["horsepower"].replace("?", pd.NA)
Auto["horsepower"] = pd.to_numeric(Auto["horsepower"])

#check out the ranges. Which feature contains much larger numbers?
# compare ranges of horsepower and weight before scaling
Auto[["horsepower", "weight"]].describe()

For algorithms based on distance, such as KNN, a variable with much larger numbers can dominate the distance calculation.

To avoid data leakage, we need to split the data first, then scale the training set, then transform test data using the parameters learned ONLY from training data.

Here is the correct workflow right away.

In [ ]:
from sklearn.model_selection import train_test_split

#We first separate the training data from the test data.
# split first so test data does not influence how training data is scaled

train_data, test_data = train_test_split(
    Auto,
    test_size=0.20,      # 20% testing and 80% training
    random_state=42      # same seed = same random split each time, making results easier to compare and troubleshoot each time code runs
)

***Note to self:*** 75/25 is the function’s default, but the lab overrides that default with `test_size=0.20`

In [ ]:
# goal- put horsepower and weight on comparable scales
# StandardScaler uses training mean and standard deviation

# Import StandardScaler to scale numerical features (mean = 0, variance = 1)
# centers each feature near 0 with standard deviation near 1
from sklearn.preprocessing import StandardScaler

# Choose the numerical features you want to scale
features = ["horsepower", "weight"]

# Instantiate the scaler object
scaler = StandardScaler()

# Fit scaler on training data (calculates mean & std) AND transform it
# learn training mean, sd to scale training values
train_scaled = scaler.fit_transform(train_data[features])

# Transform test data using the parameters learned ONLY from training data
# uses training mean, sd to scale test values
test_scaled = scaler.transform(test_data[features])

***Note to self:*** `fit_transform()` learns and applies scale; `transform()` only applies scale already learned from training

Make the Scaling Visible

In [ ]:
# convert NumPy result back to labeled DataFrame for readability
train_scaled = pd.DataFrame(
    train_scaled,
    columns=["horsepower_scaled", "weight_scaled"]
)

train_scaled.head()

In [ ]:
train_scaled.describe()

**- Are horsepower and weight still measured in their original units?**

*No, they are measured in z-scores instead of horsepower and pounds*

**- Are their numerical scales now much more comparable?**

*Yes, both features are centered near 0 with a standard deviation near 1, so weight no longer dominates solely because its original numbers were larger*

### Part 9 — Quick Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(Auto["horsepower"], Auto["mpg"])

plt.xlabel("Horsepower")
plt.ylabel("MPG")
plt.title("Horsepower vs. MPG")

plt.show()

Describe the relationship in one sentence.

**As horsepower increases, MPG (miles per gallon) generally decreases, showing a negative relationship**